In [18]:
!pip install -U pypdf langchain_community chromadb langchain langchain_openai openai tiktoken rank_bm25 sentence_transformers cohere langchain_cohere

In [39]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
import os
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain.docstore.document import Document
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from langchain_cohere import CohereRerank
import cohere
from langchain.document_loaders import PyPDFLoader
from google.colab import drive
from langchain.vectorstores import Chroma
import chromadb
from langchain.retrievers.merger_retriever import MergerRetriever
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

In [3]:
OPENAI_API_TOKEN=userdata.get('OPENAI_API_KEY')
COHERE_API_KEY = userdata.get('COHERE_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_TOKEN
os.environ["COHERE_API_KEY"] = COHERE_API_KEY

In [ ]:
drive.mount('/content/drive')

# Load Document

In [10]:
loader_harrypotter  = PyPDFLoader("/content/harrypotter.pdf")
document_harrypotter = loader_harrypotter.load()
print(len(document_harrypotter))

7


In [12]:
loader_friends = PyPDFLoader("/content/friends.pdf")
document_friends = loader_friends.load()
print(len(document_friends))

23


# Chunking

In [13]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [15]:
text_harrypotter = text_splitter.split_documents(document_harrypotter)
print(len(text_harrypotter))

101


In [16]:
text_friends = text_splitter.split_documents(document_friends)
print(len(text_friends))

124


In [17]:
embeddings = OpenAIEmbeddings()

In [20]:
os.getcwd()
CURRENT_DIR = os.path.dirname(os.path.abspath("."))
CURRENT_DIR

'/'

In [21]:
DB_DIR = os.path.join(CURRENT_DIR, "/content/db")
DB_DIR

'/content/db'

#Ingestion

In [22]:
client_settings = chromadb.config.Settings(
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False,
)

In [23]:
harrypotter_vectorstore = Chroma.from_documents(text_harrypotter,
                                       embeddings,
                                       client_settings=client_settings,
                                       collection_name="harrypotter",
                                       collection_metadata={"hnsw":"cosine"},
                                       persist_directory="/store/harrypotter")

In [24]:
friends_vectorstore = Chroma.from_documents(text_friends,
                                       embeddings,
                                       client_settings=client_settings,
                                       collection_name="friends",
                                       collection_metadata={"hnsw":"cosine"},
                                       persist_directory="/store/friends")

# Create Retrievers

In [25]:
retriever_harrypotter = harrypotter_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

In [26]:
retriever_friends = friends_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

# Merge Retrievers/ LOTR (Lord of Retriever)

In [30]:
lotr = MergerRetriever(retrievers=[retriever_harrypotter, retriever_friends])
for chunks in lotr.invoke("Who were Harry Potter's best friends? Who was Chandler Bing's girlfriend?"):
  print(chunks.page_content)

Extraverted Gryffindors, agreeable Hufflepuffs, clever 
Ravenclaws, and manipulative Slytherins. Personality and 
Individual Differences, 83(10), 174-178.
Dempster, S., oliver, A., Sunderland, J., & Thistlethwaite, J. 
(2016). What has Harry Potter Done for Me? Children’s 
Reflections on their Potter Experience. Children’ s Literature 
in Education , 47(2), 268-269.
Edwards, G. (1999). Addiction and the truth in magic realism. 
Lancet, 3(13), 354.
Popular TV Discourse: The Case of Friends  12 
 Relationships and dating.  I found 6 references to present relationships and 
dating and two that made reference to past relation ships. The latter belong to episode 
3, whereas the others were scattered in all three e pisodes. Here I provide examples of 
conversations about relationships: 
 Episode 1: 
Monica: (To Chandler) Hey sweetie. 
Chandler: Hi sweetie. So, what was with all the whispering? 
Monica: I can't tell you. It's a secret.
people in reality would finally learn their lessons at th

In [32]:
from langchain.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
)
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain.retrievers import ContextualCompressionRetriever
from langchain.document_transformers import LongContextReorder
from re import search

In [33]:
filter = EmbeddingsRedundantFilter(embeddings=embeddings)
reordering = LongContextReorder()
pipeline = DocumentCompressorPipeline(transformers=[filter, reordering])
compression_retriever_reordered = ContextualCompressionRetriever(
    base_compressor=pipeline, base_retriever=lotr,search_kwargs={"k": 3, "include_metadata": True}
)

# Generation

In [38]:
llm_model = ChatOpenAI(model_name="gpt-4o-mini")

In [40]:
qa = RetrievalQA.from_chain_type(
      llm=llm_model,
      chain_type="stuff",
      retriever = compression_retriever_reordered,
      return_source_documents = True
)

In [50]:
query ="Who were Harry Potter's best friends?? Give some of the traits Harry Potter admired about them."
results = qa.invoke(query)

In [51]:
results['result']

"Harry Potter's best friends were Ron Weasley and Hermione Granger. Some of the traits Harry admired about them include Ron's loyalty and sense of humor, as well as Hermione's intelligence and resourcefulness. Their friendship provided Harry with support and a sense of belonging, distinguishing them from more malevolent characters like Voldemort."

In [49]:
results['source_documents']

[_DocumentWithState(metadata={'author': 'elpatronhimself', 'creationdate': '2010-11-13T23:44:24-05:00', 'creator': 'PDFCreator Version 1.0.1', 'keywords': '', 'moddate': '2010-11-13T23:44:24-05:00', 'page': 13, 'page_label': '14', 'producer': 'GPL Ghostscript 8.71', 'source': '/content/friends.pdf', 'subject': '', 'title': 'Friends - AERA Paper', 'total_pages': 23}, page_content="(Pointing to the picture.)  \nMonica: Oh my God yes! Who is she? \nChandler: Julie Grath, my camp girlfriend. \nMonica: Did you break up with her? \nChandler: (pause) No, we're still together. Yeah we went out for two summers, \nand then I broke up with her. \n \nMonica: Lewis Posin! He was my best friend in fifth grade, and-and then one day \nI asked him to be my boyfriend and he said no. Do you know why? \nChandler: Because you kept talking to him while he was trying to go to the \nbathroom?!", state={'embedded_doc': [-0.0016069120028987527, 0.006057132035493851, 0.03778619319200516, -0.01443401724100113, -0

In [52]:
query ="How was Chandler Bing's relationship with Monica?? Give some dialogues to prove their relationship"
results = qa.invoke(query)

In [53]:
results['result']

'Chandler Bing\'s relationship with Monica Geller evolves significantly throughout "Friends," showcasing a mix of humor, affection, and deep emotional connection. Here are a few dialogues from the context that illustrate their relationship:\n\n1. **Affectionate Terms**:\n   - **Monica**: (To Chandler) "Hey sweetie."\n   - **Chandler**: "Hi sweetie."\n\n   This exchange demonstrates their affection for each other, using pet names indicative of a close romantic relationship.\n\n2. **Playful Banter**:\n   - **Chandler**: "Of course it is." (Mouths to Ross) "Wow -- whoa!"\n   - **Monica**: "No! But because he thought I was too faaaaa...."\n\n   This shows their playful dynamic, where they tease each other, indicating comfort and familiarity.\n\n3. **Past Relationships**:\n   - **Monica**: "I-I really think that you should apologize to Julie."\n   - **Chandler**: "Honey, are you kidding? That was like 16 years ago."\n\n   This conversation reflects their trust and involvement in discussing 

In [54]:
for source in  results["source_documents"]:
    print(source.metadata)

{'author': 'elpatronhimself', 'creationdate': '2010-11-13T23:44:24-05:00', 'creator': 'PDFCreator Version 1.0.1', 'keywords': '', 'moddate': '2010-11-13T23:44:24-05:00', 'page': 12, 'page_label': '13', 'producer': 'GPL Ghostscript 8.71', 'source': '/content/friends.pdf', 'subject': '', 'title': 'Friends - AERA Paper', 'total_pages': 23}
{'author': 'elpatronhimself', 'creationdate': '2010-11-13T23:44:24-05:00', 'creator': 'PDFCreator Version 1.0.1', 'keywords': '', 'moddate': '2010-11-13T23:44:24-05:00', 'page': 10, 'page_label': '11', 'producer': 'GPL Ghostscript 8.71', 'source': '/content/friends.pdf', 'subject': '', 'title': 'Friends - AERA Paper', 'total_pages': 23}
{'author': 'elpatronhimself', 'creationdate': '2010-11-13T23:44:24-05:00', 'creator': 'PDFCreator Version 1.0.1', 'keywords': '', 'moddate': '2010-11-13T23:44:24-05:00', 'page': 15, 'page_label': '16', 'producer': 'GPL Ghostscript 8.71', 'source': '/content/friends.pdf', 'subject': '', 'title': 'Friends - AERA Paper', 't